# Notebook 7: MDPs and value iteration (solving a world you know)

Step 7 of the ladder. Bandits (Notebook 6) had one state and no future. Now actions **change where you are**, and rewards arrive **later**. That is a **Markov decision process (MDP)**, the formal setting behind every RL algorithm you've seen (Qmaze, CartPole, Q-learning, A2C).

This notebook does something deliberately different from the earlier RL notebooks: it assumes you **know the rules** of the world exactly (what every action does, with what probability, with what reward). Then you don't need to *learn* anything from trial and error. You can **compute** the best possible behavior. Why bother? Because (a) it gives you the ground truth to grade learning algorithms against, and (b) **Q-learning turns out to be this same computation done by sampling** instead of by knowing the rules. That is the punchline, and Part 7 proves it numerically.

You need: expectations (Notebook 5) and the Q-update (Notebooks 1 and 6). NumPy and matplotlib only; runs in seconds.

### What you'll be able to say at the end
- What an MDP is, written as arrays you can print.
- What the **Bellman equation** says and how to turn it into a loop (**value iteration**).
- Why value iteration converges, how fast (the role of `gamma`), and why information moves **backward** from the reward.
- How the **order** you update states changes the work (your walk-backwards idea, in its planning form).
- How **Q-learning = value iteration with the average replaced by one sample.**
- Why tables stop working for CartPole.

Same markers as before: `WHAT / WHY / SAME AS / BECOMES / WATCH` comments, **PREDICT** cells, a knob ledger, and a sandbox.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# WHAT: plot styling only (same quiet look as earlier notebooks). Nothing here affects the math.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, MUTED, GRIDC = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlesize": 11, "axes.labelsize": 10, "font.size": 10,
    "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])   # light = low value, dark = high value

def tag(ax, x, y, text, color, dx=4, dy=0, ha="left"):
    # WHAT: write a small label next to a line instead of making you decode a legend.
    ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points", color=color, fontsize=9, va="center", ha=ha)

print("ready")

ready


# Part 1: An MDP, written as arrays

> **You are here:** the definition, in code. An MDP has five parts:
> 1. **States** `s`: where you can be.
> 2. **Actions** `a`: what you can do.
> 3. **Transitions** `P[s, a, s']`: the probability of landing in `s'` after doing `a` in `s`. (This is the world's randomness from Notebook 5, Part 4.)
> 4. **Rewards** `R[s, a, s']`: what you get for that transition.
> 5. **Discount** `gamma`: how much later rewards count (Notebook 5, Part 7).

The word **Markov** means the next state and reward depend **only on the current state and action**, not on how you got there. If they secretly depend on history, you need to put that history *into the state*. (That's exactly the Qmaze lesson: the "revisit penalty" depended on which cells you'd already visited, so a position-only picture wasn't a proper state.)

The smallest world we'll use is the slippery hallway from Notebook 5, now with a real choice: cells 0 to 4, the goal at cell 4 (the episode ends there). Two actions: **left** and **right**. Either one works with probability 0.8 and otherwise you stay put. Each step costs -0.04; arriving at the goal pays +1. At the left wall, "left" just keeps you in cell 0.

In [2]:
class MDP:
    # WHAT: everything about a world, as arrays. P[s, a, s2] = chance of landing in s2; R[s, a, s2] = reward for that transition.
    def __init__(self, P, R, done, gamma, labels=None):
        self.P, self.R, self.done, self.gamma = P, R, done.astype(bool), gamma
        self.S, self.A = P.shape[0], P.shape[1]
        self.labels = labels
        assert np.allclose(P.sum(axis=2), 1.0), "every (state, action) row of P must add to 1"   # WHAT: sanity check: outcomes are a distribution (Notebook 5, Part 1)

def build_hallway(n=5, p_move=0.8, gamma=0.9):
    S, A = n, 2                                    # actions: 0 = left, 1 = right
    P, R = np.zeros((S, A, S)), np.zeros((S, A, S))
    for s in range(n - 1):                         # the last cell (n-1) is the goal: no moves out of it
        for a, step in ((0, -1), (1, +1)):
            target = min(max(s + step, 0), n - 1)  # where a successful move lands (the wall keeps you in place)
            P[s, a, target] += p_move; P[s, a, s] += 1 - p_move
            for s2 in range(n):
                R[s, a, s2] = -0.04 + (1.0 if s2 == n - 1 and s2 != s else 0.0)   # -0.04 per step, +1 for arriving at the goal
    P[n - 1, :, n - 1] = 1.0                       # the goal absorbs: once there, you stay there (with reward 0 and value 0)
    done = np.zeros(n, bool); done[n - 1] = True
    return MDP(P, R, done, gamma)

hall = build_hallway()
print("states:", hall.S, "  actions: 2 (0 = left, 1 = right)   gamma:", hall.gamma)
print()
print("transition table P[s, a, :] for 'right' (each row adds to 1):")
for s in range(hall.S - 1):
    print(f"   from cell {s}:", np.round(hall.P[s, 1], 2))
print("rewards R[2, right, :]:", np.round(hall.R[2, 1], 2), "  (staying costs -0.04; moving to cell 3 costs -0.04; the other cells are unreachable in one move)")

states: 5   actions: 2 (0 = left, 1 = right)   gamma: 0.9

transition table P[s, a, :] for 'right' (each row adds to 1):
   from cell 0: [0.2 0.8 0.  0.  0. ]
   from cell 1: [0.  0.2 0.8 0.  0. ]
   from cell 2: [0.  0.  0.2 0.8 0. ]
   from cell 3: [0.  0.  0.  0.2 0.8]
rewards R[2, right, :]: [-0.04 -0.04 -0.04 -0.04  0.96]   (staying costs -0.04; moving to cell 3 costs -0.04; the other cells are unreachable in one move)


**What to notice.** From cell 2, "right" lands in cell 3 with probability 0.8 and in cell 2 (a slip) with probability 0.2. The full description of the world is just two arrays, `P` and `R`, plus `gamma`. Any algorithm in this notebook uses only these arrays. A **policy** is a rule for choosing actions: here, a table `pi[s, a]` of probabilities (Notebook 5, Part 1).

# Part 2: How good is a policy? (the Bellman expectation equation)

> **You are here:** before finding the *best* policy, measure a *given* one. The **value** of a state under a policy, `V(s)`, is the expected discounted return if you start there and follow the policy.

The key sentence (the **Bellman equation**): *the value of here equals the expected [reward of the next step plus gamma times the value of wherever I land].* In symbols:

`V(s) = sum over a of pi(a|s) * sum over s' of P(s'|s,a) * ( R(s,a,s') + gamma * V(s') )`

It defines `V` in terms of `V`, so you can't just read the answer off. Two ways to solve it, which must agree: **iterate** (start with `V = 0` and apply the equation over and over until nothing changes), or solve the equations **exactly** as a linear system (as in Notebook 5, Part 4).

In [3]:
def q_from_v(mdp, V):
    # WHAT: Q[s, a] = sum over s2 of P(s2|s,a) * ( R(s,a,s2) + gamma * V(s2) ).  "The expected reward of one step plus the discounted value of where I land."
    # WHY:  this one line is the Bellman equation's right-hand side. Terminal states are worth 0, so their V is forced to 0.
    Vt = V * (~mdp.done)
    return (mdp.P * (mdp.R + mdp.gamma * Vt[None, None, :])).sum(axis=2)

def policy_eval_iterative(mdp, pi, tol=1e-10, max_sweeps=100_000):
    V = np.zeros(mdp.S)
    for sweep in range(1, max_sweeps + 1):
        V_new = (pi * q_from_v(mdp, V)).sum(axis=1)             # WHAT: V(s) = average over the policy's actions of Q(s, a)
        V_new[mdp.done] = 0.0
        change = np.max(np.abs(V_new - V)); V = V_new
        if change < tol: return V, sweep
    return V, max_sweeps

def policy_eval_exact(mdp, pi):
    # WHAT: V = r_pi + gamma * P_pi V   ->   (I - gamma P_pi) V = r_pi   (a linear system, solved directly)
    P_pi = np.einsum("sa,sat->st", pi, mdp.P); r_pi = np.einsum("sa,sat,sat->s", pi, mdp.P, mdp.R)
    nd = ~mdp.done
    V = np.zeros(mdp.S)
    V[nd] = np.linalg.solve(np.eye(nd.sum()) - mdp.gamma * P_pi[np.ix_(nd, nd)], r_pi[nd])
    return V

def one_hot_policy(mdp, actions):
    pi = np.zeros((mdp.S, mdp.A)); pi[np.arange(mdp.S), actions] = 1.0; return pi

policies = {"always right": one_hot_policy(hall, np.ones(hall.S, int)),
            "always left": one_hot_policy(hall, np.zeros(hall.S, int)),
            "random 50/50": np.full((hall.S, hall.A), 0.5)}
print(" policy        | V(0..3) by iterating                | V(0..3) exact                       | sweeps | biggest difference")
for name, pi in policies.items():
    Vi, sw = policy_eval_iterative(hall, pi); Ve = policy_eval_exact(hall, pi)
    print(f" {name:13s} | {str(np.round(Vi[:4], 4)):35s} | {str(np.round(Ve[:4], 4)):35s} | {sw:5d}  | {np.max(np.abs(Vi - Ve)):.1e}")
print()
print("check 1: 'always right' matches Notebook 5's exact answer [0.4982 0.6229 0.765 0.9268]")
print("check 2: 'always left' never reaches the goal, so every cell is worth -0.04 per step forever: -0.04 / (1 - 0.9) =", -0.04 / (1 - 0.9))

 policy        | V(0..3) by iterating                | V(0..3) exact                       | sweeps | biggest difference
 always right  | [0.4982 0.6229 0.765  0.9268]       | [0.4982 0.6229 0.765  0.9268]       |    21  | 1.9e-11
 always left   | [-0.4 -0.4 -0.4 -0.4]               | [-0.4 -0.4 -0.4 -0.4]               |   189  | 9.0e-10
 random 50/50  | [-0.1032 -0.0207  0.1671  0.5124]   | [-0.1032 -0.0207  0.1671  0.5124]   |   121  | 5.2e-10

check 1: 'always right' matches Notebook 5's exact answer [0.4982 0.6229 0.765 0.9268]
check 2: 'always left' never reaches the goal, so every cell is worth -0.04 per step forever: -0.04 / (1 - 0.9) = -0.4000000000000001


**What to notice.**
- The iterative method (sweep until nothing changes) and the exact linear solve agree to rounding error for all three policies. They are the same answer, reached two ways.
- "Always right" reproduces Notebook 5's exact numbers, and "always left" gives exactly -0.4 (= -0.04 / (1 - 0.9)): it never reaches the goal, so it pays the step cost forever, discounted.
- The 50/50 random policy lands in between, and its values rise toward the goal: even a policy with no plan is worth more nearer the cheese.

# Part 3: The best policy: value iteration

> **You are here:** you can score any policy. Now find the best one without trying them all.

Replace the policy's *average* over actions with a **max**: *the value of a state under the best behavior is the best, over actions, of [expected reward plus gamma times value of where I land].* That's the **Bellman optimality equation**:

`V*(s) = max over a of  sum over s' of P(s'|s,a) * ( R(s,a,s') + gamma * V*(s') )`

**Value iteration** is just: start `V = 0`, apply the right-hand side to every state, repeat. No policy is stored. After it settles, the best action in each state is whichever one achieves that max. Watch it on the hallway, one sweep at a time:

In [4]:
def value_iteration(mdp, tol=1e-10, max_sweeps=100_000, record=False):
    V = np.zeros(mdp.S); hist, deltas = [V.copy()], []
    for sweep in range(1, max_sweeps + 1):
        V_new = q_from_v(mdp, V).max(axis=1)                    # WHAT: THE update: V(s) <- max over actions of Q(s, a). Policy evaluation used an average; this uses a max.
        V_new[mdp.done] = 0.0
        delta = np.max(np.abs(V_new - V)); V = V_new
        deltas.append(delta)
        if record: hist.append(V.copy())
        if delta < tol: break
    return V, sweep, np.array(deltas), hist

V_star, sweeps, deltas, hist = value_iteration(hall, record=True)
print(" sweep |   V(0)     V(1)     V(2)     V(3)   | biggest change this sweep")
for k in (0, 1, 2, 3, 4, 5, 10, 20, len(hist) - 1):
    if k >= len(hist): continue
    print(f"  {k:3d}  | " + "  ".join(f"{v:7.4f}" for v in hist[k][:4]) + f"  | {deltas[k - 1] if k > 0 else float('nan'):.2e}")
print()
print(f"converged after {sweeps} sweeps. best values V* = {np.round(V_star[:4], 4)}")
Q_star = q_from_v(hall, V_star)
print("Q*(s, left), Q*(s, right) for cells 0..3:")
for s in range(4): print(f"   cell {s}:  left {Q_star[s, 0]:+.4f}   right {Q_star[s, 1]:+.4f}   -> best action: {['left', 'right'][int(np.argmax(Q_star[s]))]}")

 sweep |   V(0)     V(1)     V(2)     V(3)   | biggest change this sweep
    0  |  0.0000   0.0000   0.0000   0.0000  | nan
    1  | -0.0400  -0.0400  -0.0400   0.7600  | 7.60e-01
    2  | -0.0760  -0.0760   0.5000   0.8968  | 5.40e-01
    3  | -0.1084   0.3063   0.6957   0.9214  | 3.82e-01
    4  |  0.1610   0.5160   0.7487   0.9259  | 2.69e-01
    5  |  0.3605   0.5919   0.7614   0.9267  | 1.99e-01
   10  |  0.4979   0.6229   0.7650   0.9268  | 8.06e-04
   20  |  0.4982   0.6229   0.7650   0.9268  | 3.33e-10
   21  |  0.4982   0.6229   0.7650   0.9268  | 7.05e-11

converged after 21 sweeps. best values V* = [0.4982 0.6229 0.765  0.9268]
Q*(s, left), Q*(s, right) for cells 0..3:
   cell 0:  left +0.4084   right +0.4982   -> best action: right
   cell 1:  left +0.4308   right +0.6229   -> best action: right
   cell 2:  left +0.5462   right +0.7650   -> best action: right
   cell 3:  left +0.6776   right +0.9268   -> best action: right


**What to notice.**
- Value iteration settled in 21 sweeps and the best action is **right** in every cell. Policy evaluation asks "how good is this policy?"; value iteration asks "how good is the *best* policy?" The only change in the code is `max` over actions in place of an average under a policy.
- Even in cell 0, the best-action margin is small (+0.4982 for right vs +0.4084 for left): because the floor is slippery, "left" still gets there often enough to look decent. The Q-values carry that detail; the greedy action alone hides it.

# Part 4: Why it converges, and what gamma does

> **You are here:** you saw it settle. Is that guaranteed, and how fast?

Each sweep, the error in `V` is multiplied by **at most `gamma`**. (Reason: the update averages `gamma * V` of next states, so any error in `V` shrinks by `gamma` when passed through one step.) That is called a **contraction**. Check it numerically, and test what `gamma` does to the number of sweeps:

In [5]:
Vs = V_star
errs = np.array([np.max(np.abs(h - Vs)) for h in hist])
ratios = errs[1:] / errs[:-1]
ratios = ratios[errs[:-1] > 1e-6]
print(f"error shrinks each sweep by a factor between {ratios.min():.3f} and {ratios.max():.3f}   (gamma = {hall.gamma}: the guaranteed worst case)")

print()
print("sweeps for value iteration to settle (tolerance 1e-6):")
print("  gamma | 5-cell hallway | 30-cell corridor | 'steps ahead' 1/(1-gamma)")
gamma_sweeps, gamma_sweeps_long = {}, {}
for g in (0.5, 0.9, 0.95, 0.99):
    gamma_sweeps[g] = value_iteration(build_hallway(gamma=g), tol=1e-6)[1]
    gamma_sweeps_long[g] = value_iteration(build_hallway(n=30, p_move=0.9, gamma=g), tol=1e-6)[1]
    print(f"  {g:5} |      {gamma_sweeps[g]:4d}      |       {gamma_sweeps_long[g]:4d}       |   {1 / (1 - g):6.1f}")

error shrinks each sweep by a factor between 0.228 and 0.869   (gamma = 0.9: the guaranteed worst case)

sweeps for value iteration to settle (tolerance 1e-6):
  gamma | 5-cell hallway | 30-cell corridor | 'steps ahead' 1/(1-gamma)
    0.5 |        11      |         19       |      2.0
    0.9 |        15      |         43       |     10.0
   0.95 |        16      |         44       |     20.0
   0.99 |        16      |         46       |    100.0


**What to notice.**
- Each sweep shrinks the biggest error by a factor of at most gamma (the table above showed ratios 0.228 to 0.869 at gamma 0.9). That guarantee is why value iteration always converges, from any starting guess.
- On the 5-cell hallway, sweeps went 11, 15, 16, 16 as gamma rose from 0.5 to 0.99: it flattens, because the world is only 5 cells long, so looking 100 steps ahead has nothing more to see. On the 30-cell corridor it went 19, 43, 44, 46: a big jump from 0.5 to 0.9, then nearly flat. **Sweeps did not grow in proportion to 1/(1-gamma)** in either world. What gamma really changes is *what is optimal* (how far ahead the plan looks), more than how long planning takes; that effect needs a world where the far future matters, which I did not build here.

# Part 5: Your Qmaze, solved exactly

> **You are here:** the same machinery on the maze from the Qmaze notebook. States are the open cells; actions are left, up, right, down. Rewards as in that notebook: **-0.04** per move, **-0.75** for bumping a wall (you stay put), **+1** for reaching the cheese (the episode ends). `gamma = 0.95`.

Two simplifications, so this is a true MDP: I leave out the "revisit penalty" and the "lose below a score" rule from the Qmaze notebook, because both depend on history, and a Markov state can't see history. I also include only cells that can reach the cheese (the sealed pocket at the bottom-left can't be entered, as in the Qmaze notebook's fix).

In [6]:
GRID7 = [[1, 1, 0, 1, 1, 1, 1],
         [0, 1, 0, 1, 0, 0, 1],
         [1, 1, 1, 1, 0, 1, 1],
         [1, 0, 0, 0, 0, 1, 0],
         [1, 1, 1, 1, 0, 1, 1],
         [0, 0, 0, 1, 1, 1, 0],
         [1, 1, 1, 0, 0, 1, 1]]                      # SAME AS: the 7x7 maze in the Qmaze notebook. Start = top-left, cheese = bottom-right.
LEFT, UP, RIGHT, DOWN = 0, 1, 2, 3
MOVES = {LEFT: (0, -1), UP: (-1, 0), RIGHT: (0, 1), DOWN: (1, 0)}
ARROWS = {LEFT: "←", UP: "↑", RIGHT: "→", DOWN: "↓"}

def build_maze_mdp(grid, slip=0.0, gamma=0.95):
    g = np.array(grid); n, m = g.shape; goal = (n - 1, m - 1)
    seen, stack = {goal}, [goal]                     # WHAT: flood fill backward from the cheese: which open cells can reach it at all?
    while stack:
        r, c = stack.pop()
        for dr, dc in MOVES.values():
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < m and g[nr, nc] == 1 and (nr, nc) not in seen:
                seen.add((nr, nc)); stack.append((nr, nc))
    cells = sorted(seen); index = {c: i for i, c in enumerate(cells)}; S = len(cells)
    P, R = np.zeros((S, 4, S)), np.zeros((S, 4, S))
    for (r, c), i in index.items():
        if (r, c) == goal:
            P[i, :, i] = 1.0; continue
        for a in range(4):
            for a2 in range(4):                      # WHAT: with probability `slip` the floor sends you in a random direction instead
                prob = (1 - slip) * (a2 == a) + slip / 4
                if prob == 0: continue
                dr, dc = MOVES[a2]; nr, nc = r + dr, c + dc
                if not (0 <= nr < n and 0 <= nc < m and g[nr, nc] == 1):
                    P[i, a, i] += prob; R[i, a, i] += prob * -0.75        # bump into a wall: stay put, -0.75
                else:
                    j = index[(nr, nc)]
                    P[i, a, j] += prob; R[i, a, j] += prob * (1.0 if (nr, nc) == goal else -0.04)
    # R as stored above is probability-weighted per target; convert to a per-transition reward so that  P * R  sums correctly
    with np.errstate(divide="ignore", invalid="ignore"):
        R = np.where(P > 0, R / P, 0.0)
    done = np.zeros(S, bool); done[index[goal]] = True
    mdp = MDP(P, R, done, gamma); mdp.cells, mdp.index, mdp.goal, mdp.grid = cells, index, goal, g
    return mdp

maze = build_maze_mdp(GRID7)
Vm, sweeps_m, deltas_m, _ = value_iteration(maze, tol=1e-10)
Qm = q_from_v(maze, Vm); pol_m = Qm.argmax(axis=1)
start = maze.index[(0, 0)]
print(f"{maze.S} states, 4 actions.  value iteration settled in {sweeps_m} sweeps.  V*(start) = {Vm[start]:.4f}")

def follow(mdp, policy, start_cell=(0, 0), cap=200):
    s = mdp.index[start_cell]; path = [mdp.cells[s]]
    for _ in range(cap):
        if mdp.done[s]: break
        a = policy[s]; nxt = np.argmax(mdp.P[s, a]); s = nxt; path.append(mdp.cells[s])    # deterministic maze: follow the one possible landing
    return path
path_m = follow(maze, pol_m)
print(f"following the greedy policy from the start reaches the cheese in {len(path_m) - 1} moves  (the shortest route in the Qmaze notebook was 14)")
g_ = maze.gamma
hand = sum(-0.04 * g_ ** t for t in range(13)) + 1.0 * g_ ** 13       # WHAT: by hand: 13 steps cost -0.04 each, then the 14th move arrives at the cheese for +1; discount each by gamma^t
print(f"check by hand: 13 steps x (-0.04) and then +1.0 on the 14th move, each discounted: {hand:.4f}   (value iteration said {Vm[start]:.4f})")

28 states, 4 actions.  value iteration settled in 15 sweeps.  V*(start) = 0.1240
following the greedy policy from the start reaches the cheese in 14 moves  (the shortest route in the Qmaze notebook was 14)
check by hand: 13 steps x (-0.04) and then +1.0 on the 14th move, each discounted: 0.1240   (value iteration said 0.1240)


In [7]:
def draw_maze_values(mdp, V, policy, path=None, title=""):
    n, m = mdp.grid.shape
    img = np.full((n, m), np.nan)
    for (r, c), i in mdp.index.items(): img[r, c] = V[i]
    fig, ax = plt.subplots(figsize=(6.4, 5.8))
    ax.imshow(np.where(mdp.grid == 0, 0.0, np.nan), cmap="Greys", vmin=0, vmax=3, alpha=0.35)     # walls (and sealed pockets) in grey
    ax.imshow(img, cmap=SEQ, vmin=np.nanmin(img) - 0.1, vmax=np.nanmax(img))
    for (r, c), i in mdp.index.items():
        if mdp.done[i]: ax.text(c, r, "C", ha="center", va="center", fontsize=14, color="white", fontweight="bold"); continue
        dark = (V[i] - np.nanmin(img)) / (np.nanmax(img) - np.nanmin(img) + 1e-9) > 0.55
        ax.text(c, r - 0.12, ARROWS[policy[i]], ha="center", va="center", fontsize=14, color="white" if dark else INK)
        ax.text(c, r + 0.28, f"{V[i]:.2f}", ha="center", va="center", fontsize=7, color="white" if dark else MUTED)
    if path:
        ax.plot([c for r, c in path], [r for r, c in path], color=ORANGE, linewidth=2.2, alpha=0.9)
        ax.text(0, -0.55, "start", color=ORANGE, fontsize=9, ha="center")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); ax.set_title(title)
    for sp in ax.spines.values(): sp.set_visible(False)
    plt.show()

draw_maze_values(maze, Vm, pol_m, path_m, "Optimal values (darker = better) and best action in every cell; orange = route from the start")

<Figure size 704x638 with 1 Axes>

**What to notice.**
- Value iteration settled in 15 sweeps. The arrows form the 14-move route the Qmaze notebook found by learning, and the start's value matches the hand calculation (13 steps at -0.04, then +1, discounted = 0.1240). That is a check that the MDP encoding really means what we intended.
- Values rise smoothly toward the cheese: it is the same picture a trained Q-table is trying to learn, but computed exactly in a fraction of a second.
- Caveat: this planner needs the maze's rules (`P` and `R`) handed to it. Q-learning (Part 7) needs only the ability to take steps.

### A slippery floor: does the best plan change?
The Qmaze notebook had a `slip` setting. Now you can answer exactly, with no learning noise: build the maze with different slip probabilities (with probability `slip`, the move goes in a random direction), solve each by value iteration, and compare. Also take the policy that was optimal with *no* slip and **evaluate it** on each slippery floor (Part 2's exact policy evaluation).

In [8]:
base_pol = pol_m                                                         # WHAT: the best policy for the NO-slip maze
print(" slip | V*(start) when planning for it | V(start) if you keep the no-slip plan | cells where the best action differs from the no-slip plan")
slip_rows = {}
for sl in (0.0, 0.1, 0.3, 0.5, 0.7):
    mz = build_maze_mdp(GRID7, slip=sl)
    Vs_, _, _, _ = value_iteration(mz, tol=1e-10)
    pol_s = q_from_v(mz, Vs_).argmax(axis=1)
    V_keep = policy_eval_exact(mz, one_hot_policy(mz, base_pol))
    differ = int(np.sum((pol_s != base_pol) & ~mz.done))
    slip_rows[sl] = (Vs_[mz.index[(0, 0)]], V_keep[mz.index[(0, 0)]], differ)
    print(f" {sl:4} |            {slip_rows[sl][0]:+8.3f}            |              {slip_rows[sl][1]:+8.3f}              |                    {differ:2d} of {mz.S - 1}")

 slip | V*(start) when planning for it | V(start) if you keep the no-slip plan | cells where the best action differs from the no-slip plan
  0.0 |              +0.124            |                +0.124              |                     0 of 27
  0.1 |              -0.328            |                -0.328              |                     0 of 27
  0.3 |              -1.453            |                -1.453              |                     0 of 27
  0.5 |              -2.986            |                -2.986              |                     0 of 27
  0.7 |              -5.060            |                -5.060              |                     0 of 27


**What to notice.**
- Slip costs a lot of value: V*(start) falls from +0.124 to -5.06 as slip rises to 0.7.
- But in every cell, **the best action never changed** (0 of 27 differ at every slip level I tried), so keeping the no-slip plan lost nothing, and the two value columns match exactly. This is *not* a bug: with this kind of slip (a random move, equally likely in any direction) nothing makes a different route safer, so the shortest route stays best. A slip that is *not* uniform, such as a slippery patch next to a pit, would break this. I did not test that case.
- The practical lesson: a model error can leave the policy unchanged yet change the value a lot, so judge the policy and the value separately.

# Part 6: Your walk-backwards idea, as a planning question

> **You are here:** you said you'd "hold pointers in place and fill them in", working backward from the cheese instead of letting values creep. In planning (where the rules are known) that idea has a precise form: **the order in which you update the states within a sweep.**

Value iteration so far updated every state from the *previous* sweep's values (a "synchronous" sweep). But you can update **in place**: use each new value immediately. Then the *order* matters. If you update cells **in order of distance from the cheese**, outward, then when you reach a cell its better neighbor was updated a moment ago, so value can race from the cheese all the way back in a single sweep. If you go the wrong way, it creeps one cell per sweep.

**PREDICT first.** On the 7x7 maze, the cheese is the *last* cell in row-major order (top-left to bottom-right). How many sweeps will each of these need to settle: (a) synchronous, (b) in-place in row-major order, (c) in-place in *reverse* row-major order, (d) in-place outward from the cheese by actual distance? Type a guess for each, then run.

In [9]:
# PREDICT: sweeps needed -- (a) synchronous ___  (b) in place, row-major ___  (c) in place, reverse row-major ___  (d) in place, outward from the cheese ___
my_prediction = ""
print("your prediction:", my_prediction if my_prediction else "(blank: type one above first)")

your prediction: (blank: type one above first)


In [10]:
def goal_distances(mdp):
    # WHAT: how many moves from each state to the goal at best? (backward breadth-first search over "can move from s to s2" links)
    dist = np.full(mdp.S, np.inf); g = np.where(mdp.done)[0][0]; dist[g] = 0; frontier = [g]
    reach = (mdp.P > 0).any(axis=1) & ~np.eye(mdp.S, dtype=bool)               # reach[s, s2]: some action can take you from s to a DIFFERENT state s2
    while frontier:
        nxt = []
        for s2 in frontier:
            for s in np.where(reach[:, s2])[0]:
                if dist[s] == np.inf: dist[s] = dist[s2] + 1; nxt.append(s)
        frontier = nxt
    return dist

def converge_inplace(mdp, order_fn, tol=1e-9, max_sweeps=5000, v0=0.0):
    # WHAT: value iteration where each state's new value is used IMMEDIATELY (in place). order_fn decides which states to update, and in what order, each sweep.
    V = np.full(mdp.S, float(v0)); V[mdp.done] = 0.0; backups = 0       # v0 = the starting guess for every non-goal state (0 unless you say otherwise)
    for sweep in range(1, max_sweeps + 1):
        change = 0.0
        for s in order_fn(mdp, sweep, V):
            q = (mdp.P[s] * (mdp.R[s] + mdp.gamma * (V * (~mdp.done))[None, :])).sum(axis=1)     # WHAT: one "backup": Q(s, a) for the 4 actions, from the CURRENT V
            new = q.max(); change = max(change, abs(new - V[s])); V[s] = new; backups += 1
        if change < tol: return sweep, backups, V
    return max_sweeps, backups, V

non_goal = lambda mdp: [s for s in range(mdp.S) if not mdp.done[s]]
def order_row_major(mdp, sweep, V): return non_goal(mdp)
def order_reverse(mdp, sweep, V): return non_goal(mdp)[::-1]
def order_outward(mdp, sweep, V):
    d = goal_distances(mdp); return sorted(non_goal(mdp), key=lambda s: d[s])
def order_inward(mdp, sweep, V):
    d = goal_distances(mdp); return sorted(non_goal(mdp), key=lambda s: -d[s])
def order_random(mdp, sweep, V): return list(np.random.default_rng(sweep).permutation(non_goal(mdp)))   # WHAT: a fresh shuffle each sweep, but seeded by the sweep number so every run gives the same numbers

sync_sweeps = value_iteration(maze, tol=1e-9)[1]
print(f"{'update order':44s}{'sweeps':>8s}{'state backups':>16s}")
print(f"{'synchronous (all from the previous sweep)':44s}{sync_sweeps:8d}{sync_sweeps * (maze.S - 1):16d}")
order_results = {"synchronous": (sync_sweeps, sync_sweeps * (maze.S - 1))}
for label, fn in (("in place, row-major", order_row_major), ("in place, reverse row-major", order_reverse),
                  ("in place, outward from the cheese", order_outward), ("in place, inward (farthest first)", order_inward),
                  ("in place, random order each sweep", order_random)):
    sw, bk, V_ = converge_inplace(maze, fn)
    order_results[label] = (sw, bk)
    print(f"{label:44s}{sw:8d}{bk:16d}    (max error vs exact answer: {np.max(np.abs(V_ - Vm)):.1e})")

update order                                  sweeps   state backups
synchronous (all from the previous sweep)         15             405
in place, row-major                               14             378    (max error vs exact answer: 0.0e+00)
in place, reverse row-major                        5             135    (max error vs exact answer: 0.0e+00)
in place, outward from the cheese                  2              54    (max error vs exact answer: 0.0e+00)
in place, inward (farthest first)                 15             405    (max error vs exact answer: 0.0e+00)
in place, random order each sweep                 10             270    (max error vs exact answer: 0.0e+00)


**What to notice.** This is the planning-world version of your walk-backwards idea: the order of updates, with the same final answer every time (the error column is 0).
- **Outward from the cheese** (value flows the way information flows): 2 sweeps, 54 backups, against 15 sweeps and 405 backups for the synchronous version. **Inward** (farthest first) is no better than synchronous (15 / 405). Row-major sits near the middle (14 / 378) mostly by accident of how the maze is laid out; reverse row-major did well (5 / 135) because the cheese is at the bottom right.
- On the 30-cell corridor the same pattern holds (outward and reverse 24 sweeps; row-major and inward 49), and on the slippery maze it shrinks (31 to 46 sweeps): when moves can go backward, the good news has to flow both ways and order helps less.
- Random order is in the middle of the pack. That is one seed; I did not average over many.
- Your instinct was right for planning: update where the information starts. The open question for Notebook 8 is whether this survives when you must **sample** the world instead of reading its rules.

# Part 7: Q-learning is value iteration with the average replaced by one sample

> **You are here:** the punchline. Compare the two updates for one state-action pair `(s, a)`:

- **Value iteration (knows the rules):** `Q(s,a) <- sum over s' of P(s'|s,a) * ( R + gamma * max over a' of Q(s',a') )`. It needs `P`.
- **Q-learning (doesn't):** take action `a`, **observe one** next state `s'` and reward `r`, then `Q(s,a) <- Q(s,a) + alpha * ( r + gamma * max Q(s',a') - Q(s,a) )`. It needs only experience.

The claim: the Q-learning target, **on average over the randomness of the world, equals the value-iteration backup.** So Q-learning is value iteration where the exact average is replaced by a single noisy sample, and the step size `alpha` smooths out the noise (Notebook 5's square-root law). Verify the first claim by brute force on the slippery maze, starting from an arbitrary `Q`:

In [11]:
slippy = build_maze_mdp(GRID7, slip=0.3)
rng = np.random.default_rng(0)
Q_arb = rng.normal(0, 1, (slippy.S, 4))                                  # WHAT: an arbitrary, wrong Q-table (the claim holds for ANY table)
s, a = slippy.index[(2, 1)], RIGHT

exact_backup = np.sum(slippy.P[s, a] * (slippy.R[s, a] + slippy.gamma * Q_arb.max(axis=1) * (~slippy.done)))   # WHAT: value iteration's backup (uses P)
n_samples = 400_000
nxt = rng.choice(slippy.S, size=n_samples, p=slippy.P[s, a])              # WHAT: act in the world n times: only the OUTCOMES are observed
targets = slippy.R[s, a, nxt] + slippy.gamma * Q_arb.max(axis=1)[nxt] * (~slippy.done[nxt])   # WHAT: Q-learning's one-sample target r + gamma * max Q(s2, .)
print(f"value-iteration backup (exact average, uses P) : {exact_backup:+.5f}")
print(f"average of {n_samples:,d} sampled Q-learning targets      : {targets.mean():+.5f}   (standard error {targets.std() / np.sqrt(n_samples):.5f})")
print(f"spread of ONE sampled target (std)             : {targets.std():.3f}   <- this is the noise each Q-learning update carries")

value-iteration backup (exact average, uses P) : +1.62864
average of 400,000 sampled Q-learning targets      : +1.62881   (standard error 0.00061)
spread of ONE sampled target (std)             : 0.386   <- this is the noise each Q-learning update carries


**What to notice.** The sampled average matches the exact backup to within its standard error. So each Q-learning update is an **unbiased but noisy** version of a value-iteration backup, and **the noise is the whole difference.** That is why Q-learning needs small `alpha` or many visits (Notebook 6, Part 9: the constant-step trade-off) while value iteration can take full steps.

Now watch the sampled version converge to the exact answer. Run Q-learning on the **hallway MDP** while acting **completely at random** (uniform actions): it never follows its own best guess, yet should still find the *optimal* Q. Measure the largest error against the exact `Q*` from Part 3.

In [12]:
def q_learning_hallway(mdp, alpha_rule, n_steps=30_000, seed=0, record_every=250):
    r = np.random.default_rng(seed)
    Q = np.zeros((mdp.S, mdp.A)); counts = np.zeros((mdp.S, mdp.A)); s = 0
    errs = []
    nd = ~mdp.done
    for t in range(1, n_steps + 1):
        a = r.integers(mdp.A)                                              # WHAT: act at random: pure exploration (off-policy learning)
        s2 = r.choice(mdp.S, p=mdp.P[s, a]); reward = mdp.R[s, a, s2]      # WHAT: the world answers with ONE sampled outcome
        target = reward + mdp.gamma * (0.0 if mdp.done[s2] else Q[s2].max())
        counts[s, a] += 1
        alpha = alpha_rule(counts[s, a])
        Q[s, a] += alpha * (target - Q[s, a])                              # SAME AS: Notebook 1 Part 8: one gradient step on 0.5*(target - Q)^2
        s = 0 if mdp.done[s2] else s2                                      # WHAT: episode over -> restart at cell 0
        if t % record_every == 0: errs.append(np.max(np.abs(Q[nd] - Q_star[nd])))
    return Q, np.array(errs)

rules = {"alpha = 0.5": lambda n: 0.5, "alpha = 0.1": lambda n: 0.1, "alpha = 0.02": lambda n: 0.02, "alpha = 1/n (per cell and action)": lambda n: 1.0 / n}
ql = {k: q_learning_hallway(hall, rule) for k, rule in rules.items()}
print(" step size                          | biggest |Q - Q*| after 30,000 random steps | greedy policy matches the optimal one?")
for k, (Q_, errs_) in ql.items():
    same = bool(np.array_equal(Q_[:4].argmax(axis=1), Q_star[:4].argmax(axis=1)))
    print(f" {k:34s} |                 {errs_[-1]:.4f}                          | {same}")

 step size                          | biggest |Q - Q*| after 30,000 random steps | greedy policy matches the optimal one?
 alpha = 0.5                        |                 0.0739                          | True
 alpha = 0.1                        |                 0.0398                          | True
 alpha = 0.02                       |                 0.0076                          | True
 alpha = 1/n (per cell and action)  |                 0.0484                          | True


In [13]:
fig, ax = plt.subplots(figsize=(7.8, 4.2))
steps_axis = np.arange(1, len(next(iter(ql.values()))[1]) + 1) * 250
for (k, (Q_, errs_)), col in zip(ql.items(), (ORANGE, BLUE, AQUA, YELLOW)):
    ax.plot(steps_axis, errs_, color=col)
    tag(ax, steps_axis[-1], errs_[-1], k.replace(" (per cell and action)", ""), col, dx=5)
ax.set_yscale("log"); ax.set_xlim(0, 41000); ax.set_xlabel("random steps taken"); ax.set_ylabel("biggest error vs the exact Q*")
ax.set_title("Q-learning (sampled) approaching value iteration's exact answer")
plt.show()

<Figure size 858x462 with 1 Axes>

**What to notice.**
- The sampled target averages 1.62881 against the exact backup 1.62864 (standard error 0.00061): **unbiased**, but one sample has a spread of 0.386, which is the noise each update carries.
- With purely random actions, all four step-size rules still recovered the **optimal** greedy policy: Q-learning is off-policy, so it can learn the best plan from clumsy experience.
- The error is a trade-off. alpha = 0.5 stays noisy (0.074); 0.1 and 0.02 are smaller (0.040, 0.0076); 1/n starts slowest and is still falling at 30,000 steps (0.048). These are single runs read at the final step, and the curves bounce, so the ordering of 0.1 against 1/n could change with another seed.

# Part 8: Why the table breaks on CartPole

> **You are here:** tables and exact solutions are lovely, but how far do they scale?

Value iteration touches every state in every sweep, and Q-learning keeps one number per state-action. CartPole's state is **4 continuous numbers** (cart position, cart speed, pole angle, pole speed). The earlier notebooks handled it by chopping each number into **buckets**. Count how many table cells that makes:

In [14]:
print(" buckets per number | table states (b^4) | Q-table cells (2 actions) | memory at 8 bytes each")
for b in (6, 10, 20, 50, 100):
    states = b ** 4
    mem = states * 2 * 8
    unit = "KB" if mem < 1e6 else "MB" if mem < 1e9 else "GB"
    val = mem / (1e3 if unit == "KB" else 1e6 if unit == "MB" else 1e9)
    print(f"        {b:4d}        | {states:>18,d} | {states * 2:>25,d} | {val:8.1f} {unit}")
print()
print("the 7x7 maze has", maze.S, "states.  The slippery hallway has", hall.S, ".")

 buckets per number | table states (b^4) | Q-table cells (2 actions) | memory at 8 bytes each
           6        |              1,296 |                     2,592 |     20.7 KB
          10        |             10,000 |                    20,000 |    160.0 KB
          20        |            160,000 |                   320,000 |      2.6 MB
          50        |          6,250,000 |                12,500,000 |    100.0 MB
         100        |        100,000,000 |               200,000,000 |      1.6 GB

the 7x7 maze has 28 states.  The slippery hallway has 5 .


**What to notice.** Each extra bucket per number multiplies the table by `b` four times over, so fine buckets get expensive fast (the **curse of dimensionality**). And a table can't *generalize*: knowing the value of one bucket says nothing about its neighbor, so every cell must be visited. The remedy is Notebook 4: replace the table with a **neural network** that maps a state to values, so nearby states share what they learn. That is the whole step from tabular Q-learning to DQN, and from value iteration to "fitted" methods. Two more caveats: CartPole's true transition probabilities `P` aren't something we know, so exact planning isn't available there anyway, and a network trained this way can **fail to converge** where the table was guaranteed to. That price is real and is the subject of Notebook 9.

# Part 9: Sandbox (your ideas compete here)

> **You are here:** race **update orders** for in-place value iteration. This is the planning version of your walk-backwards idea, with a fair ruler.

**The ruler.** An order function `my_order(mdp, sweep, V)` returns the list of states to update this sweep, in order. The harness runs in-place value iteration with it until a whole sweep changes nothing (tolerance `1e-9`) and reports **sweeps** and **backups** (the total number of single-state updates, the real cost). Three worlds: a **30-cell corridor** with the goal at the far end (moves succeed 90% of the time), the **7x7 maze**, and the **7x7 maze with 30% slip**. The order may look at `mdp.P` (it is a planner, it knows the rules) and the current `V`, but not at the final answer.

In [15]:
def build_corridor(n=30, p_move=0.9, gamma=0.95):
    return build_hallway(n=n, p_move=p_move, gamma=gamma)        # SAME AS Part 1's hallway, just longer

worlds7 = [("corridor 30", build_corridor()), ("maze 7x7", build_maze_mdp(GRID7)), ("maze 7x7, slip 0.3", build_maze_mdp(GRID7, slip=0.3))]

def race7(entries):
    print(f"{'':36s}" + "".join(f"{nm:>26s}" for nm, _ in worlds7))
    print(f"{'':36s}" + "".join(f"{'sweeps | backups':>26s}" for _ in worlds7))
    out = {}
    for label, fn in entries:
        row = []
        for _, w in worlds7:
            sw, bk, _V = converge_inplace(w, fn); row.append((sw, bk))
        out[label] = row
        print(f"{label:36s}" + "".join(f"{f'{sw:5d}  |  {bk:7d}':>26s}" for sw, bk in row))
    return out

reference7 = [("row-major", order_row_major), ("reverse row-major", order_reverse),
              ("outward from the goal", order_outward), ("inward (farthest first)", order_inward), ("random each sweep", order_random)]
race7_out = race7(reference7)

                                                   corridor 30                  maze 7x7        maze 7x7, slip 0.3
                                              sweeps | backups          sweeps | backups          sweeps | backups
row-major                                       49  |     1421            14  |      378            45  |     1215
reverse row-major                               24  |      696             5  |      135            34  |      918
outward from the goal                           24  |      696             2  |       54            31  |      837
inward (farthest first)                         49  |     1421            15  |      405            46  |     1242
random each sweep                               41  |     1189            10  |      270            44  |     1188


**Why did the corridor need 24 sweeps even with the best order?** A guess to test: in the corridor the far cells' true values are *negative* (the cheese is so far away that discounted costs outweigh it), so starting every cell at **0** is *optimistic*, and the in-place sweep spends passes undoing wrong early preferences for unexplored neighbors. Test it: same corridor and same best order, but start every value at -3 (below any true value) instead of 0.

In [16]:
corr = worlds7[0][1]
for v0 in (0.0, -3.0):
    sw, bk, _ = converge_inplace(corr, order_outward, v0=v0)
    print(f"corridor 30, outward order, every value starts at {v0:+.0f}: {sw} sweeps, {bk} backups")
print("true values: farthest cell", np.round(value_iteration(corr, tol=1e-12)[0][0], 3), " nearest-but-one cell", np.round(value_iteration(corr, tol=1e-12)[0][-2], 3))

corridor 30, outward order, every value starts at +0: 24 sweeps, 696 backups
corridor 30, outward order, every value starts at -3: 22 sweeps, 638 backups
true values: farthest cell -0.444  nearest-but-one cell 0.95


**What to notice.**
- Outward and reverse beat the others in every world; inward and row-major are the slowest on the corridor.
- My guess for why the corridor needs 24 sweeps even in the best order was that starting values at 0 are too optimistic. The test says only a little: starting at -3 took 22 sweeps instead of 24. So that guess was mostly wrong. I have not tested the remaining candidate (the slip sending value backward along the corridor each sweep).

### Your slot
Edit `my_order` and re-run. Ideas: outward order but **skip states whose value didn't change last sweep** (that is **prioritized sweeping**: only revisit states next to something that just changed); update the states with the **biggest recent change first**; sweep outward *and then* inward in the same pass; or order by `V` itself (best-valued states first). The order function gets the current `V`, so it can be dynamic. Rule: don't peek at the exact answer.

In [17]:
def my_order(mdp, sweep, V):
    # SANDBOX: replace with your own idea. Return a list of state indices (not the goal) to update this sweep, in order.
    return non_goal(mdp)                      # <- starts as plain row-major. Try sorting by goal distance, or by V.

_ = race7(reference7 + [("MY ORDER", my_order)])

                                                   corridor 30                  maze 7x7        maze 7x7, slip 0.3
                                              sweeps | backups          sweeps | backups          sweeps | backups


row-major                                       49  |     1421            14  |      378            45  |     1215
reverse row-major                               24  |      696             5  |      135            34  |      918
outward from the goal                           24  |      696             2  |       54            31  |      837
inward (farthest first)                         49  |     1421            15  |      405            46  |     1242
random each sweep                               41  |     1189            10  |      270            44  |     1188
MY ORDER                                        49  |     1421            14  |      378            45  |     1215


# Part 10: The knob ledger, self-check, and what's next

## Knob ledger (Notebooks 1 to 7)
Families: **objective**, **model**, **search**, **experience**, **judging**. New in this notebook: gamma as a cost driver, the slip (model accuracy), sweep order, and the table size.

In [18]:
rows = [
    ("learning rate / step size (NB1)", "search", "too small: slow", "too big: blows up"),
    ("gamma", "objective", f"0.5: {gamma_sweeps_long[0.5]} sweeps (30-cell corridor), plans ~2 steps ahead", f"0.99: {gamma_sweeps_long[0.99]} sweeps, plans ~100 steps ahead"),
    ("exploration (NB6)", "search", "greedy: can lock in", "random: still finds Q* (Q-learning is off-policy)"),
    ("Q-learning step size alpha", "search", f"0.5: error {ql['alpha = 0.5'][1][-1]:.3f} (noisy)", f"0.02: error {ql['alpha = 0.02'][1][-1]:.3f}   (1/n: {ql['alpha = 1/n (per cell and action)'][1][-1]:.3f})"),
    ("slip in the world (the model)", "model", f"slip 0: V*(start) {slip_rows[0.0][0]:+.2f}", f"slip 0.7: V*(start) {slip_rows[0.7][0]:+.2f}  (cells whose best action changed: {slip_rows[0.7][2]} of 27)"),
    ("update order (planning)", "search", f"inward order: {order_results['in place, inward (farthest first)'][0]} sweeps", f"outward order: {order_results['in place, outward from the cheese'][0]} sweeps"),
    ("table size (buckets)", "model", "6 buckets/number: 1,296 states", "100 buckets/number: 100,000,000 states"),
]
print(f"{'knob':34s}{'family':12s}{'at one extreme':66s}{'at the other'}")
print("-" * 190)
for knob, fam, a, b in rows:
    print(f"{knob:34s}{fam:12s}{a:66s}{b}")

knob                              family      at one extreme                                                    at the other
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
learning rate / step size (NB1)   search      too small: slow                                                   too big: blows up
gamma                             objective   0.5: 19 sweeps (30-cell corridor), plans ~2 steps ahead           0.99: 46 sweeps, plans ~100 steps ahead
exploration (NB6)                 search      greedy: can lock in                                               random: still finds Q* (Q-learning is off-policy)
Q-learning step size alpha        search      0.5: error 0.074 (noisy)                                          0.02: error 0.008   (1/n: 0.048)
slip in the world (the model)     model       slip 0: V*(start) +0.12                          

## Self-check (answer in your own words before you read the answers)
| Question | Short answer |
|---|---|
| What are the parts of an MDP? | States, actions, transition probabilities `P`, rewards `R`, and a discount `gamma`. "Markov" means the next state depends only on the current state and action. |
| What does the Bellman equation say? | The value of a state is the expected reward of one step plus gamma times the expected value of where you land. |
| How is value iteration different from policy evaluation? | Policy evaluation averages over the policy's actions; value iteration takes the max over actions. |
| Why does value iteration converge? | Each sweep shrinks the error by at least a factor of gamma (a contraction). |
| What does gamma cost? | Larger gamma means more sweeps (information must travel farther) and, in sampled methods, noisier returns. |
| Why does information move backward? | Only the cell next to the reward "knows" about it at first; each sweep passes the knowledge one step back. Updating in the right order passes it all the way in one sweep. |
| How is Q-learning related to value iteration? | Its target equals the value-iteration backup on average; it replaces the exact average with one sampled outcome, so it needs only experience, but it is noisy. |
| Why do tables fail for CartPole? | The state space is continuous and 4-dimensional: buckets multiply, and tables can't generalize. |

## What's next
- You can now compute the exact answer for small worlds, which is the ground truth for grading any learner.
- You've seen that Q-learning is a sampled, noisy version of value iteration.

**Notebook 8: Q-learning, TD, and reverse replay.** The sampled versions, head to head: Q-learning, TD(0), Monte Carlo, n-step returns, eligibility traces, and **your walk-backwards idea** (update the steps of an episode in reverse). They race on the mazes using this notebook's exact `V*` as the answer key, and you'll find out how many laps backward replay really needs.